In [2]:
# ===========================================
# 금융상품 추천 시스템 개발 (LightGBM)
# ===========================================

# 필요한 라이브러리 import
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 머신러닝 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# LightGBM
try:
    from lightgbm import LGBMClassifier
    print("✅ LightGBM 라이브러리 import 성공!")
except ImportError as e:
    print(f"❌ Import 오류: {e}")
    print("LightGBM 설치: pip install lightgbm")

print("🚀 LightGBM 금융상품 추천 시스템")
print("=" * 50)

# 타겟 변수 이름
target_names = {
    'LIQ': '유동성자산',
    'CDS': '양도성예금증서', 
    'NMMF': '비머니마켓펀드',
    'STOCKS': '주식보유',
    'RETQLIQ': '퇴직준비금유동성'
}

# 데이터 로드
try:
    df = pd.read_csv('data/cleaned_scf_data.csv')
    X = df.iloc[:, :-5]
    y = df.iloc[:, -5:]
    print(f"✅ 실제 데이터 로드: {X.shape[0]:,}명, {X.shape[1]}개 특성")
except FileNotFoundError:
    print("※ 실제 데이터 없음. 샘플 데이터 생성 중...")
    np.random.seed(42)
    n_samples = 2000
    X = pd.DataFrame(np.random.randn(n_samples, 16), 
                     columns=[f'feature_{i+1}' for i in range(16)])
    y = pd.DataFrame(np.random.randint(0, 2, (n_samples, 5)),
                     columns=list(target_names.keys()))
    print(f"✅ 샘플 데이터 생성: {X.shape[0]:,}명, {X.shape[1]}개 특성")

print(f"데이터 준비 완료: X={X.shape}, y={y.shape}")

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"훈련 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")

# === LightGBM 기본 설정 ===
n_features = X.shape[1]
colsample_ratio = np.sqrt(n_features) / n_features  # max_features='sqrt' 대응

base_lgbm_model = LGBMClassifier(
    n_estimators=500,         # 팀원과 동일
    max_depth=20,             # 동일
    random_state=42,          # 동일
    n_jobs=-1,                # 동일
    
    # 유사 효과 매핑
    min_child_samples=5,      # min_samples_leaf=5 대응
    colsample_bytree=colsample_ratio,
    subsample=0.8,            # bootstrap 유사
    
    # 학습률
    learning_rate=0.1,
    
    # class_weight='balanced' 유사 적용
    class_weight='balanced',
    
    # 기타
    reg_alpha=0.1,  # L1
    reg_lambda=1.0, # L2
    objective='binary',  # 각 타겟 독립적인 이진 분류
    verbose=-1

)

# MultiOutputClassifier로 래핑
multilabel_lgbm = MultiOutputClassifier(base_lgbm_model)

# 모델 훈련
print("🚀 MultiOutput LightGBM 훈련 시작...")
multilabel_lgbm.fit(X_train, y_train)
print("✅ MultiOutput LightGBM 훈련 완료!")

# 예측
print("🔍 예측 시작...")
test_pred = multilabel_lgbm.predict(X_test)
print("✅ 예측 완료!")

# 성능 평가
accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred, average='samples', zero_division=0)
recall = recall_score(y_test, test_pred, average='samples', zero_division=0)
f1 = f1_score(y_test, test_pred, average='samples', zero_division=0)

print("★ LightGBM 전체 모델 성능")
print("=" * 40)
print(f"정확도 (Accuracy):  {accuracy:.3f}")
print(f"정밀도 (Precision): {precision:.3f}")
print(f"재현율 (Recall):    {recall:.3f}")
print(f"F1 점수:           {f1:.3f}")


✅ LightGBM 라이브러리 import 성공!
🚀 LightGBM 금융상품 추천 시스템
✅ 실제 데이터 로드: 22,975명, 14개 특성
데이터 준비 완료: X=(22975, 14), y=(22975, 5)
훈련 데이터: (18380, 14), 테스트 데이터: (4595, 14)
🚀 MultiOutput LightGBM 훈련 시작...
✅ MultiOutput LightGBM 훈련 완료!
🔍 예측 시작...
✅ 예측 완료!
★ LightGBM 전체 모델 성능
정확도 (Accuracy):  0.755
정밀도 (Precision): 0.924
재현율 (Recall):    0.949
F1 점수:           0.923
